# Attempt 1

## Import libraries

In [1]:
# --- Imports
import os, math, random, time
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# --- Repro
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --- Paths (adjust if needed)
DATA_DIR = "../../data"      # your CSVs
SUBMIT_PATH = "./submission.csv"

# --- Labels
NUM_LABELS = 388             # 0..387 per competition schema
LABEL_COLUMNS = [str(i) for i in range(NUM_LABELS)]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cpu


In [2]:
# Load base CSVs
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
meta  = pd.read_csv(os.path.join(DATA_DIR, "metaData.csv"))

# Ensure proper dtype to join
train["project_id"] = train["project_id"].astype(int)
val["project_id"]   = val["project_id"].astype(int)
test["project_id"]  = test["project_id"].astype(int)
meta["project_id"]  = meta["project_id"].astype(int)

# Join (left)
train_joined = train.merge(meta, on="project_id", how="left", suffixes=("_train", "_meta"))
val_joined   = val.merge(meta, on="project_id",   how="left", suffixes=("_train", "_meta"))
test_joined  = test.merge(meta, on="project_id",  how="left", suffixes=("_train", "_meta"))

# Keep only columns we actually need for this minimal baseline
cols_keep = [
    "id", "project_id", "room", "work_operation_cluster_code",
    # minimal metadata (unused in this ultra-minimal baseline; left here if you want to extend)
    "insurance_company", "office_distance", "case_creation_year", "case_creation_month",
]
train_joined = train_joined[cols_keep].copy()
val_joined   = val_joined[cols_keep].copy()
test_joined  = test_joined[["id","project_id","room","work_operation_cluster_code"]].copy()


In [3]:
ROOM_CATEGORIES = [
    "andre områder","kjøkken","stue","gang","soverom","bad","bod","vaskerom","wc","kjeller","garasje"
]
ROOM_INDEX = {r:i for i,r in enumerate(ROOM_CATEGORIES)}

def map_room(room_name: str) -> str:
    if not isinstance(room_name, str):
        return "ukjent"
    s = room_name.lower()
    for r in ROOM_CATEGORIES:
        if r in s:
            return r
    return "ukjent"

def multi_hot_from_codes(codes: List[int], num_labels: int = NUM_LABELS) -> np.ndarray:
    v = np.zeros(num_labels, dtype=np.float32)
    for c in codes:
        if 0 <= int(c) < num_labels:
            v[int(c)] = 1.0
    return v

def mask_codes_for_training(full_codes: List[int],
                            min_hide:int=1,
                            max_frac:float=0.5,
                            rng:np.random.Generator=np.random.default_rng(SEED)) -> Tuple[List[int], List[int]]:
    """
    Split a room's full code set S into observed O and hidden H (targets).
    Ensures H is non-empty if S has size > 1. If S has size == 1, return H = [] (no masking).
    """
    S = sorted(set(int(x) for x in full_codes))
    if len(S) <= 1:
        return S, []  # nothing to hide
    
    max_hide = max(min_hide, int(math.floor(len(S) * max_frac)))
    max_hide = min(max_hide, len(S)-1)  # keep at least one observed
    n_hide = rng.integers(low=min_hide, high=max_hide+1)
    H = sorted(rng.choice(S, size=n_hide, replace=False).tolist())
    O = [x for x in S if x not in H]
    return O, H


In [4]:
USE_ROOM_ONEHOT = False  # set True if you want to append ~11-dim room one-hot

def aggregate_per_id(df: pd.DataFrame) -> pd.DataFrame:
    # one row per (project_id, room, id); collect operation codes and a representative room string
    g = (
        df.groupby(["project_id", "room", "id"])["work_operation_cluster_code"]
          .apply(list).reset_index(name="codes")
    )
    # Store a canonical room_category (optional)
    g["room_category"] = g["room"].apply(map_room)
    return g

train_agg = aggregate_per_id(train_joined)
val_agg   = aggregate_per_id(val_joined)
test_agg  = aggregate_per_id(test_joined)

rng_train = np.random.default_rng(SEED)
rng_val   = np.random.default_rng(SEED + 1)

def build_xy_from_agg(df_agg: pd.DataFrame,
                      do_mask: bool,
                      rng: np.random.Generator) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """
    Returns:
        X: [N, D_in]
        Y: [N, NUM_LABELS]  (multi-label)
        meta: df with columns ['id','project_id','room_category','observed_codes','hidden_codes']
    """
    X_list, Y_list, meta_rows = [], [], []
    for _, row in df_agg.iterrows():
        rid = int(row["id"])
        codes_full = [int(c) for c in row["codes"]]
        room_cat = row["room_category"]

        if do_mask:
            observed, hidden = mask_codes_for_training(codes_full, rng=rng)
        else:
            observed, hidden = codes_full, []  # test-time: no masking; we'll predict missing later

        x_ops = multi_hot_from_codes(observed, NUM_LABELS)
        if USE_ROOM_ONEHOT:
            rc_vec = np.zeros(len(ROOM_CATEGORIES), dtype=np.float32)
            rc_idx = ROOM_INDEX.get(room_cat, None)
            if rc_idx is not None:
                rc_vec[rc_idx] = 1.0
            x_vec = np.concatenate([x_ops, rc_vec], axis=0)
        else:
            x_vec = x_ops

        y_vec = multi_hot_from_codes(hidden, NUM_LABELS)  # targets are the hidden ones

        X_list.append(x_vec)
        Y_list.append(y_vec)
        meta_rows.append({
            "id": rid,
            "project_id": int(row["project_id"]),
            "room_category": room_cat,
            "observed_codes": observed,
            "hidden_codes": hidden
        })

    X = np.stack(X_list, axis=0)
    Y = np.stack(Y_list, axis=0)
    meta = pd.DataFrame(meta_rows)
    return X, Y, meta

X_train, Y_train, meta_train = build_xy_from_agg(train_agg, do_mask=True, rng=rng_train)
X_val,   Y_val,   meta_val   = build_xy_from_agg(val_agg,   do_mask=True, rng=rng_val)

print("Train shapes:", X_train.shape, Y_train.shape)
print("Val   shapes:", X_val.shape,   Y_val.shape)


Train shapes: (185635, 388) (185635, 388)
Val   shapes: (10827, 388) (10827, 388)


In [5]:
class RoomsDataset(Dataset):
    def __init__(self, X: np.ndarray, Y: np.ndarray):
        self.X = X.astype(np.float32)
        self.Y = Y.astype(np.float32)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), torch.from_numpy(self.Y[idx])

train_ds = RoomsDataset(X_train, Y_train)
val_ds   = RoomsDataset(X_val,   Y_val)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False, drop_last=False)

INPUT_DIM  = X_train.shape[1]
OUTPUT_DIM = NUM_LABELS


In [6]:
class MLP(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)  # logits
        )
    def forward(self, x):
        return self.net(x)

model = MLP(INPUT_DIM, OUTPUT_DIM).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()   # sigmoid + binary cross-entropy
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [7]:
def run_epoch(model, loader, train:bool=True):
    model.train(mode=train)
    total_loss, total_n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss = criterion(logits, yb)
        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_n += xb.size(0)
    return total_loss / max(1,total_n)

EPOCHS = 6
for epoch in range(1, EPOCHS+1):
    tr_loss = run_epoch(model, train_loader, train=True)
    va_loss = run_epoch(model, val_loader,   train=False)
    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} | val loss {va_loss:.4f}")


Epoch 01 | train loss 0.0446 | val loss 0.0260
Epoch 02 | train loss 0.0236 | val loss 0.0216
Epoch 03 | train loss 0.0201 | val loss 0.0194
Epoch 04 | train loss 0.0183 | val loss 0.0183
Epoch 05 | train loss 0.0173 | val loss 0.0177
Epoch 06 | train loss 0.0167 | val loss 0.0173


### Tuning threshold

In [13]:
# --- 1) Get validation probabilities and bookkeeping we need for tuning
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))
from metrics.score import normalized_rooms_score

model.eval()
val_probs_list = []
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        val_probs_list.append(torch.sigmoid(logits).cpu().numpy())
probs_val = np.concatenate(val_probs_list, axis=0)  # shape: [N_val, 388]

# We also need:
#  - targets_val: list[list[int]] of the hidden codes H for each val room
#  - observed_val: list[list[int]] of the observed codes O for each val room
#  - room_category_val: list[str] if you want room-aware top-K later
targets_val = meta_val["hidden_codes"].tolist()    # from build_xy_from_agg(...)
observed_val = meta_val["observed_codes"].tolist() # from build_xy_from_agg(...)
room_category_val = meta_val["room_category"].tolist()


In [14]:
# --- 2) Threshold tuning utilities

def preds_from_probs(probs: np.ndarray,
                     thresholds: np.ndarray,
                     observed_lists: list[list[int]]) -> list[list[int]]:
    """
    Convert probability matrix -> index predictions using per-label thresholds,
    and remove any codes that are already observed for that room.
    """
    # binarize per label
    bin_mat = (probs >= thresholds[None, :]).astype(np.int32)
    # turn rows into sets of predicted indices, minus observed
    preds = []
    for i in range(bin_mat.shape[0]):
        idx = bin_mat[i].nonzero()[0].tolist()
        obs = set(int(c) for c in observed_lists[i])
        idx = [j for j in idx if j not in obs]
        preds.append(idx)
    return preds

def score_with_thresholds(probs, thresholds, observed, targets) -> float:
    preds = preds_from_probs(probs, thresholds, observed)
    return normalized_rooms_score(preds, targets)

def tune_thresholds_coordinate_descent(probs: np.ndarray,
                                       observed: list[list[int]],
                                       targets: list[list[int]],
                                       init: float = 0.5,
                                       grid = np.linspace(0.1, 0.9, 9),
                                       max_passes: int = 2) -> np.ndarray:
    """
    Greedy coordinate descent over labels.
    Iteratively tries a small grid for each label while keeping others fixed.
    Two passes is usually enough and runs in a few minutes.
    """
    ths = np.full((probs.shape[1],), init, dtype=np.float32)
    base = score_with_thresholds(probs, ths, observed, targets)
    print(f"[tune] start score: {base:.4f}")

    for p in range(max_passes):
        improved_any = False
        for j in range(probs.shape[1]):
            best_t, best_s = ths[j], base
            for t in grid:
                if t == ths[j]: 
                    continue
                ths_try = ths.copy()
                ths_try[j] = t
                s = score_with_thresholds(probs, ths_try, observed, targets)
                if s > best_s:
                    best_s, best_t = s, t
            if best_t != ths[j]:
                ths[j] = best_t
                base = best_s
                improved_any = True
        print(f"[tune] pass {p+1}: score {base:.4f}")
        if not improved_any:
            break
    return ths

# Run tuning
thresholds_vec = tune_thresholds_coordinate_descent(
    probs=probs_val,
    observed=observed_val,
    targets=targets_val,
    init=0.5,
    grid=np.linspace(0.1, 0.9, 9),
    max_passes=2
)
print("Tuned thresholds summary:",
      f"min={thresholds_vec.min():.2f}, med={np.median(thresholds_vec):.2f}, max={thresholds_vec.max():.2f}")


[tune] start score: 0.3988
[tune] pass 1: score 0.5214
[tune] pass 2: score 0.5214
Tuned thresholds summary: min=0.10, med=0.20, max=0.80


In [15]:
# Build test inputs (no masking)
X_test_ops = []
test_ids = []
observed_by_id = {}

for _, row in test_agg.iterrows():
    rid = int(row["id"])
    codes = [int(c) for c in row["codes"]]
    observed_by_id[rid] = sorted(set(codes))
    x_vec = multi_hot_from_codes(codes, NUM_LABELS)
    if USE_ROOM_ONEHOT:
        rc_vec = np.zeros(len(ROOM_CATEGORIES), dtype=np.float32)
        rc_idx = ROOM_INDEX.get(row["room_category"], None)
        if rc_idx is not None:
            rc_vec[rc_idx] = 1.0
        x_vec = np.concatenate([x_vec, rc_vec], axis=0)
    X_test_ops.append(x_vec)
    test_ids.append(rid)

X_test = np.stack(X_test_ops, axis=0).astype(np.float32)

# Predict
model.eval()
with torch.no_grad():
    logits = model(torch.from_numpy(X_test).to(DEVICE))
    probs  = torch.sigmoid(logits).cpu().numpy()

# --- 3) Apply tuned thresholds on test
model.eval()
with torch.no_grad():
    logits_test = model(torch.from_numpy(X_test).to(DEVICE))
    probs_test  = torch.sigmoid(logits_test).cpu().numpy()  # [N_test, 388]

# Use tuned per-label thresholds (fallback to 0.5 if not found)
ths = thresholds_vec if 'thresholds_vec' in globals() else np.full((NUM_LABELS,), 0.5, dtype=np.float32)

# Binarize with per-label thresholds
pred_bin = (probs_test >= ths[None, :]).astype(np.int32)

# Never predict already observed ops
for i, rid in enumerate(test_ids):
    for c in observed_by_id[rid]:
        if 0 <= c < NUM_LABELS:
            pred_bin[i, c] = 0


In [16]:
submission = pd.DataFrame(pred_bin, columns=LABEL_COLUMNS)
submission.insert(0, "id", test_ids)

# Ensure types are ints (0/1)
for c in LABEL_COLUMNS:
    submission[c] = submission[c].astype(int)

# Optional: sort by id to match sample ordering (not strictly required if all ids present)
submission = submission.sort_values("id").reset_index(drop=True)

print(submission.shape)
submission.head()
submission.to_csv(SUBMIT_PATH, index=False)
print(f"Saved: {SUBMIT_PATH}")


(18299, 389)
Saved: ./submission.csv
